## CAP182 SS3 – Part B


**Business problem:** predict whether a customer will become dormant within six months, so STADIOEquities can prioritise timely customer-retention interventions.

**Prediction date:** 1 January 2025. Features must represent information available on or before this date. The target is `dormant_within_6m` from `table9_target_dormancy.csv`.

This notebook compares logistic regression and random forest as classification methods and selects the model using held-out test performance. The preprocessing is fitted inside each modelling pipeline, after the train/test split, to prevent test-set information leaking into training.

> Important: tables 2, 3 and 4 contain aggregates measured at the later extract date as well as event timestamps. The later aggregates are deliberately not used. Event-level records are filtered to timestamps on or before 2025-01-01 before customer-level features are calculated.

## 1. Dataset loading

The client extract contains ten CSV tables. The target table contains **3,218 customer records**. In the supplied extract, 1,021 customers (31.7%) are labelled dormant within six months and 2,197 (68.3%) are labelled not dormant. This is a moderately imbalanced binary classification problem, so accuracy will be reported alongside precision, recall, F1-score and ROC-AUC.

Place the extracted `stadioequities` folder beside this notebook, or update `DATA_DIR` below.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    classification_report, ConfusionMatrixDisplay, RocCurveDisplay
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

OBSERVATION_DATE = pd.Timestamp("2025-01-01")
DATA_DIR_CANDIDATES = [
    Path("stadioequities"),
    Path("mogalelebogang_client_data/stadioequities"),
    Path("mogalelebogang_client_data") / "stadioequities",
    Path("data/stadioequities"),
]

DATA_DIR = next((p for p in DATA_DIR_CANDIDATES if p.exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find the stadioequities folder. Extract mogalelebogang_client_data.zip "
        "and set DATA_DIR to the folder containing table1_account_onboarding.csv through table10_data_quality_metadata.csv."
    )

def load_table(filename):
    path = DATA_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")
    return pd.read_csv(path)

account = load_table("table1_account_onboarding.csv")
funding = load_table("table2_funding_balance.csv")
behaviour = load_table("table3_app_web_behaviour.csv")
trading = load_table("table4_trading_activity.csv")
products = load_table("table5_product_subscription.csv")
demographics = load_table("table6_demographics_profile.csv")
support = load_table("table7_support_feedback.csv")
marketing = load_table("table8_marketing_engagement.csv")
target = load_table("table9_target_dormancy.csv")
metadata = load_table("table10_data_quality_metadata.csv")

print("Loaded tables:")
for name, df in [
    ("account", account), ("funding", funding), ("behaviour", behaviour),
    ("trading", trading), ("products", products), ("demographics", demographics),
    ("support", support), ("marketing", marketing), ("target", target),
    ("metadata", metadata)
]:
    print(f"{name:14s}: {df.shape[0]:>6,} rows × {df.shape[1]} columns")

target["observation_date"] = pd.to_datetime(target["observation_date"], errors="coerce")
target["dormant_within_6m"] = target["dormant_within_6m"].astype(str).str.strip().str.lower().map(
    {"true": 1, "false": 0, "1": 1, "0": 0, "yes": 1, "no": 0}
)
target = target.dropna(subset=["customer_id", "dormant_within_6m"])
print("\nTarget distribution:")
print(target["dormant_within_6m"].value_counts().rename(index={0: "Not dormant", 1: "Dormant within 6 months"}))
print("Positive-class proportion:", round(target["dormant_within_6m"].mean(), 3))

In [ ]:
# ---------- Table 1: onboarding/account information ----------
a = account.copy()
for col in ["registration_date", "kyc_completion_date", "first_deposit_date"]:
    a[col] = pd.to_datetime(a[col], errors="coerce")
a = a.drop_duplicates(subset=["customer_id"], keep="first")

# Keep dates and derived durations only where available by the observation date.
a["registered_by_observation"] = (a["registration_date"] <= OBSERVATION_DATE).astype("int8")
a["days_since_registration_at_observation"] = (
    OBSERVATION_DATE - a["registration_date"]
).dt.days
a["kyc_completed_by_observation"] = (
    a["kyc_completion_date"].notna() & (a["kyc_completion_date"] <= OBSERVATION_DATE)
).astype("int8")
a["first_deposit_by_observation"] = (
    a["first_deposit_date"].notna() & (a["first_deposit_date"] <= OBSERVATION_DATE)
).astype("int8")
a["days_to_first_deposit_safe"] = a["days_to_first_deposit"].where(
    a["first_deposit_date"] <= OBSERVATION_DATE
)
account_features = a[[
    "customer_id", "account_status", "kyc_status", "onboarding_completion",
    "abandoned_step", "onboarding_duration_minutes", "account_type",
    "registered_by_observation", "days_since_registration_at_observation",
    "kyc_completed_by_observation", "first_deposit_by_observation",
    "days_to_first_deposit_safe"
]].copy()

# ---------- Table 2: funding events; ignore later extract-date aggregates ----------
f = funding.copy()
f["deposit_date"] = pd.to_datetime(f["deposit_date"], errors="coerce")
f["withdrawal_date"] = pd.to_datetime(f["withdrawal_date"], errors="coerce")
dep = f[f["deposit_date"].notna() & (f["deposit_date"] <= OBSERVATION_DATE)].copy()
withd = f[f["withdrawal_date"].notna() & (f["withdrawal_date"] <= OBSERVATION_DATE)].copy()

dep_features = dep.groupby("customer_id").agg(
    deposits_pre_obs=("deposit_date", "size"),
    deposit_amount_sum_pre_obs=("deposit_amount", "sum"),
    deposit_amount_mean_pre_obs=("deposit_amount", "mean"),
    deposit_last_date=("deposit_date", "max")
)
dep_features["days_since_last_deposit_at_observation"] = (
    OBSERVATION_DATE - dep_features["deposit_last_date"]
).dt.days
dep_features = dep_features.drop(columns=["deposit_last_date"])

withdraw_features = withd.groupby("customer_id").agg(
    withdrawals_pre_obs=("withdrawal_date", "size"),
    withdrawal_amount_sum_pre_obs=("withdrawal_amount", "sum"),
    withdrawal_amount_mean_pre_obs=("withdrawal_amount", "mean"),
    withdrawal_last_date=("withdrawal_date", "max")
)
withdraw_features["days_since_last_withdrawal_at_observation"] = (
    OBSERVATION_DATE - withdraw_features["withdrawal_last_date"]
).dt.days
withdraw_features = withdraw_features.drop(columns=["withdrawal_last_date"])

# ---------- Table 3: app/web events; exclude later repeated aggregates ----------
b = behaviour.copy()
b["session_date"] = pd.to_datetime(b["session_date"], errors="coerce")
b = b[b["session_date"].notna() & (b["session_date"] <= OBSERVATION_DATE)].copy()
behaviour_features = b.groupby("customer_id").agg(
    sessions_pre_obs=("session_date", "size"),
    mean_session_duration_pre_obs=("session_duration", "mean"),
    mean_screens_viewed_pre_obs=("screens_viewed", "mean"),
    distinct_feature_categories_pre_obs=("feature_usage", "nunique"),
    onboarding_sessions_event_mean=("onboarding_sessions", "mean"),
    abandonment_count_event_mean=("abandonment_count", "mean"),
    most_recent_session=("session_date", "max")
)
behaviour_features["days_since_last_session_at_observation"] = (
    OBSERVATION_DATE - behaviour_features["most_recent_session"]
).dt.days
behaviour_features = behaviour_features.drop(columns=["most_recent_session"])
# Categorical mode from eligible events
feature_mode = b.groupby("customer_id")["feature_usage"].agg(
    lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan
).rename("most_used_feature_pre_obs")
device_mode = b.groupby("customer_id")["device_type"].agg(
    lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan
).rename("main_device_pre_obs")
behaviour_features = behaviour_features.join([feature_mode, device_mode])

# ---------- Table 4: trading events; exclude later repeated aggregates ----------
tr = trading.copy()
tr["trade_date"] = pd.to_datetime(tr["trade_date"], errors="coerce")
tr = tr[tr["trade_date"].notna() & (tr["trade_date"] <= OBSERVATION_DATE)].copy()
trading_features = tr.groupby("customer_id").agg(
    trades_pre_obs=("trade_date", "size"),
    trade_value_sum_pre_obs=("trade_value", "sum"),
    trade_value_mean_pre_obs=("trade_value", "mean"),
    instruments_held_mean_pre_obs=("instruments_held", "mean"),
    concentration_mean_pre_obs=("concentration_measure", "mean"),
    last_trade_date=("trade_date", "max")
)
trading_features["days_since_last_trade_at_observation"] = (
    OBSERVATION_DATE - trading_features["last_trade_date"]
).dt.days
trading_features = trading_features.drop(columns=["last_trade_date"])
trade_type_mode = tr.groupby("customer_id")["trade_type"].agg(
    lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan
).rename("most_common_trade_type_pre_obs")
instrument_mode = tr.groupby("customer_id")["instrument_type"].agg(
    lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan
).rename("most_common_instrument_pre_obs")
trading_features = trading_features.join([trade_type_mode, instrument_mode])

# ---------- Table 5: product subscription state as at observation date ----------
p = products.copy()
for col in ["premium_start_date", "premium_cancellation_date"]:
    p[col] = pd.to_datetime(p[col], errors="coerce")
p = p.drop_duplicates(subset=["customer_id"], keep="first")
p["premium_started_by_observation"] = (
    p["premium_start_date"].notna() & (p["premium_start_date"] <= OBSERVATION_DATE)
).astype("int8")
p["premium_cancelled_by_observation"] = (
    p["premium_cancellation_date"].notna() &
    (p["premium_cancellation_date"] <= OBSERVATION_DATE)
).astype("int8")
product_features = p[[
    "customer_id", "premium_status", "product_count", "product_type",
    "product_take_up", "cancellation_count",
    "premium_started_by_observation", "premium_cancelled_by_observation"
]].copy()

# ---------- Table 6: demographic/profile table ----------
d = demographics.drop_duplicates(subset=["customer_id"], keep="first").copy()

# ---------- Table 7: support tickets; only tickets before/on observation date ----------
s = support.copy()
s["ticket_date"] = pd.to_datetime(s["ticket_date"], errors="coerce")
s = s[s["ticket_date"].notna() & (s["ticket_date"] <= OBSERVATION_DATE)].copy()
support_features = s.groupby("customer_id").agg(
    support_tickets_pre_obs=("ticket_date", "size"),
    mean_resolution_time_pre_obs=("resolution_time", "mean"),
    mean_feedback_score_pre_obs=("feedback_score", "mean"),
    last_ticket_date=("ticket_date", "max")
)
support_features["days_since_last_ticket_at_observation"] = (
    OBSERVATION_DATE - support_features["last_ticket_date"]
).dt.days
support_features = support_features.drop(columns=["last_ticket_date"])
ticket_reason_mode = s.groupby("customer_id")["ticket_reason"].agg(
    lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan
).rename("most_common_ticket_reason_pre_obs")
support_features = support_features.join(ticket_reason_mode)

# ---------- Table 8: marketing records available by observation date ----------
m = marketing.copy()
m["campaign_date"] = pd.to_datetime(m["campaign_date"], errors="coerce")
m = m[m["campaign_date"].isna() | (m["campaign_date"] <= OBSERVATION_DATE)].copy()
m = m.drop_duplicates(subset=["customer_id"], keep="first")
marketing_features = m[[
    "customer_id", "acquisition_channel", "acquisition_cost",
    "email_sent", "email_opened", "email_clicked", "nudge_received", "nudge_engagement"
]].copy()

# ---------- Assemble one row per customer ----------
features = target[["customer_id", "observation_date", "dormant_within_6m"]].copy()
for part in [
    account_features, dep_features.reset_index(), withdraw_features.reset_index(),
    behaviour_features.reset_index(), trading_features.reset_index(),
    product_features, d, support_features.reset_index(), marketing_features
]:
    features = features.merge(part, on="customer_id", how="left")

# Keep only the observation date and target from target table; never use outcome-detail columns.
features = features.drop(columns=["observation_date"])
features = features.drop_duplicates(subset=["customer_id"])
y = features["dormant_within_6m"].astype(int)
X = features.drop(columns=["customer_id", "dormant_within_6m"])

# Normalise inconsistent categorical casing and whitespace.
for col in X.select_dtypes(include=["object"]).columns:
    X[col] = X[col].apply(
        lambda value: str(value).strip().lower()
        if pd.notna(value) and str(value).strip().lower() not in {"", "nan", "none"}
        else np.nan
    ).astype(object)

# Convert booleans to numeric 0/1 to avoid mixed boolean/object types.
for col in X.columns:
    if X[col].dtype == "bool":
        X[col] = X[col].astype("int8")

print("Customer-level feature matrix:", X.shape)
print("Target rows:", len(y))
print("Duplicate customer rows:", features["customer_id"].duplicated().sum())
print("Feature columns:", len(X.columns))
display(X.head())

### Preprocessing rationale

- **Missing values:** median imputation for numeric variables and most-frequent imputation for categorical variables.
- **Categorical features:** one-hot encoding, with unknown categories ignored so inference does not fail on new categories.
- **Scaling:** numeric variables are standardised for logistic regression. Scaling is harmless for the random forest pipeline and keeps the preprocessing structure consistent.
- **Leakage prevention:** preprocessing is fitted only on the training subset inside the pipeline.
- **Identifier exclusion:** `customer_id` is used only for joins and is excluded from `X`.
- **Time cut-off:** event rows after 1 January 2025 are not used to create behavioural features.

## 3. Train/test split and model training

A stratified 80/20 split is used to preserve the target ratio in training and test sets. `random_state=42` makes the split reproducible. Both models use balanced class weights to account for the 31.7% positive class. Logistic regression is retained as an interpretable baseline; random forest can model non-linear effects and interactions. The random forest uses 300 trees and a controlled minimum leaf size to reduce overfitting risk.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = [c for c in X_train.columns if c not in numeric_features]

numeric_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_preprocessor, numeric_features),
    ("cat", categorical_preprocessor, categorical_features)
])

models = {
    "Logistic Regression": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            class_weight="balanced", max_iter=2000, random_state=42
        ))
    ]),
    "Random Forest": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=300, max_depth=None, min_samples_leaf=3,
            class_weight="balanced_subsample", random_state=42, n_jobs=-1
        ))
    ])
}

training_outputs = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    training_outputs[name] = {
        "model": model,
        "train_pred": model.predict(X_train),
        "test_pred": model.predict(X_test),
        "test_prob": model.predict_proba(X_test)[:, 1]
    }
    print(f"{name} trained.")
print(f"Training rows: {len(X_train):,}; test rows: {len(X_test):,}")

Random Forest trained.
Training rows: 2,574; test rows: 644


## 4. Model validation on unseen data

The held-out test subset is not used to fit preprocessing or model parameters. The following metrics measure classification performance on this unseen subset. F1 balances precision and recall; ROC-AUC measures ranking quality across thresholds. The confusion matrix helps quantify false positives and false negatives, which have different business consequences for retention outreach.

In [ ]:
results = []
for name, out in training_outputs.items():
    pred, prob = out["test_pred"], out["test_prob"]
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1-score": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, prob)
    })

results_df = pd.DataFrame(results).sort_values(
    ["F1-score", "ROC-AUC"], ascending=False
).reset_index(drop=True)
display(results_df.style.format({
    "Accuracy": "{:.3f}", "Precision": "{:.3f}", "Recall": "{:.3f}",
    "F1-score": "{:.3f}", "ROC-AUC": "{:.3f}"
}))

best_name = results_df.iloc[0]["Model"]
best_model = training_outputs[best_name]["model"]
best_pred = training_outputs[best_name]["test_pred"]
best_prob = training_outputs[best_name]["test_prob"]
print("Selected model based on held-out F1-score, with ROC-AUC as tie-breaker:", best_name)
print("\nClassification report:")
print(classification_report(y_test, best_pred, target_names=["Not dormant", "Dormant within 6 months"], zero_division=0))